In [1]:
# import the required libraries
import sys
from pathlib import Path

import dask.array as da
import ipywidgets as widgets
import napari
from czitools.metadata_tools import czi_metadata as czimd
from czitools.read_tools import read_tools as czird
from czitools.utils import misc
from ipyfilechooser import FileChooser
from IPython.display import HTML, display
from napari.utils.colormaps import Colormap

# Make the shared utility importable from any workshop subdirectory.
for utility_dir in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (utility_dir / "workshop_utils.py").is_file():
        if str(utility_dir) not in sys.path:
            sys.path.insert(0, str(utility_dir))
        break
else:
    raise FileNotFoundError("Start from the workshop root or a subdirectory.")

from workshop_utils import initialize_workshop

workshop_root, project_dir, data_dir, model_dir = initialize_workshop(
    "czi_handling"
)


Kernel Python: f:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\.pixi\envs\default\python.exe
Workshop: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11
Project: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\examples\czi_handling
Data: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data
Model: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data\DL_models


In [2]:
# choose local file
fc = FileChooser()
fc.default_path = str(data_dir)
fc.filter_pattern = '*.czi'
display(fc)
# Select a CZI file and confirm it before running the next cell.

FileChooser(path='F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data', filename='', title='', show_hidden=F…

In [4]:
# get only specific metadata
if fc.selected is None:
    raise ValueError("Select a CZI file in the file chooser and confirm it before running this cell.")
filepath = str(fc.selected)

# get the complete metadata at once as one big class
mdata = czimd.CziMetadata(filepath)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdata, reduced_params=True)

# create a ipywdiget to show the dataframe with the metadata
wd = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd:
    display(HTML(mdframe.to_html()))
display(widgets.VBox(children=[wd]))

  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


In [5]:
# return array with dimension order STCZYX(A)
array6d, mdata= czird.read_6darray(filepath)

# print the shape of the array etc.
print(f"Shape: {array6d.shape}")
print(f"Dimensions: {array6d.dims}")

for k, v in array6d.attrs.items():
    print(f"{k} :  {v}")

# show dask array structure
if isinstance(array6d, da.Array):
    print(array6d)
else:
    print("Shape:", array6d.shape, "dtype:", array6d.dtype)


  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


2026-11-12 16:35:59,279 - czitools - INFO - read_6darray: Reading 30 planes (S=1, T=3, C=2, Z=5)


  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


Shape: (1, 3, 2, 5, 170, 240)
Dimensions: ('S', 'T', 'C', 'Z', 'Y', 'X')
description :  6D image data from CZI file
source :  F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data\T=3_Z=5_CH=2_X=240_Y=170.czi
axes :  STCZYX
subset_planes :  {'S': (0, 0), 'T': (0, 2), 'C': (0, 1), 'Z': (0, 4)}
Shape: (1, 3, 2, 5, 170, 240) dtype: uint16


In [6]:
# show in napari (requires napari to be installed!)
viewer = napari.Viewer()

# loop over all channels
for ch in range(0, array6d.sizes["C"]):

    # extract channel subarray by positional index (isel) to avoid
    # KeyError when C coordinate labels are not 0-based integers
    sub_array = array6d.isel(C=ch)

    # get the scaling factors for that channel and adapt Z-axis scaling
    scalefactors = [1.0] * len(sub_array.shape)
    scalefactors[sub_array.get_axis_num("Z")] = mdata.scale.ratio["zx_sf"]# * 1.00001

    # remove the last scaling factor in case of an RGB image
    if "A" in sub_array.dims:
        # remove the A axis from the scaling factors
        scalefactors.pop(sub_array.get_axis_num("A"))

    # get colors and channel name
    chname = mdata.channelinfo.names[ch]

    # inside the CZI metadata_tools colors are defined as ARGB hexstring
    rgb = "#" + mdata.channelinfo.colors[ch][3:]
    ncmap = Colormap(["#000000", rgb], name="cm_" + chname)

    # add the channel to the viewer
    viewer.add_image(
        sub_array,
        name=chname,
        colormap=ncmap,
        blending="additive",
        scale=scalefactors,
        gamma=0.85,
    )

    # set the axis labels based on the dimensions
    viewer.dims.axis_labels = sub_array.dims


In [ ]:
napari.utils.nbscreenshot(viewer)